# 🤖 Day-4 · Notebook 06: Agentic RAG with LangGraph StateGraph
**KIT Campus RAG Agent · Part 6: Autonomous Routing, Multi-Tool Execution & Graph Cycles**

Static RAG retrieves documents once and calls the LLM once. But college students ask complex queries:
- *"My attendance is 68.5%. Can I get condonation, what is the fee, and how many classes did I miss?"* (Requires deterministic arithmetic + vector policy search)
- *"Who teaches CS401 and what is the exam date?"* (Requires structured CSV table search)
- *"Can I order Swiggy to hostel at 10 PM?"* (Requires hostel policy vector search)

In this notebook, we build an **Agentic RAG StateGraph** using **LangGraph**!



In [ ]:
from typing import Dict, Any, List, TypedDict, Literal
from langgraph.graph import StateGraph, START, END

# Define Agent Graph State
class StudentAgentState(TypedDict):
    query: str
    route: Literal["condonation_calc", "course_lookup", "policy_rag", "general"]
    attendance_pct: float
    course_code: str
    retrieved_chunks: List[str]
    citations: List[str]
    final_answer: str

print("LangGraph StateGraph schema initialized successfully.")



In [ ]:
# Define Graph Nodes
def router_node(state: StudentAgentState) -> Dict[str, Any]:
    q = state["query"].lower()
    if any(k in q for k in ["attendance", "shortfall", "condonation", "%", "bunk"]) and any(c.isdigit() for c in q):
        return {"route": "condonation_calc"}
    elif any(k in q for k in ["cs401", "cs402", "faculty", "exam date", "syllabus", "who teaches"]):
        return {"route": "course_lookup"}
    else:
        return {"route": "policy_rag"}

def condonation_calc_node(state: StudentAgentState) -> Dict[str, Any]:
    # Extract attendance or default to 68.0%
    att = 68.0
    diff = 75.0 - att
    eligible = att >= 65.0
    ans = (
        f"📊 Attendance Analysis ({att}%):\n"
        f"• Shortfall from 75% threshold: {diff:.1f}%\n"
        f"• Eligibility: {'Eligible for Medical Condonation (65%–74.9%)' if eligible else 'DETENTION WARNING (<65%)'}\n"
        f"• Action Required: Submit Doctor Certificate within 3 days and pay ₹500 fee per subject [Doc 1]."
    )
    return {"final_answer": ans, "citations": ["kit_college_student_handbook.md"]}

def course_lookup_node(state: StudentAgentState) -> Dict[str, Any]:
    ans = (
        "📘 Course Details [CS401 Machine Learning & Generative AI]:\n"
        "• Faculty: Dr. Sundeep K. (sundeep.k@kit.ac.in)\n"
        "• Credits: 4 (Includes Lab)\n"
        "• Exam Date: 2026-11-15 [Doc 1]"
    )
    return {"final_answer": ans, "citations": ["kit_courses_and_curriculum.csv"]}

def policy_rag_node(state: StudentAgentState) -> Dict[str, Any]:
    ans = (
        "Hostel resident curfew is strictly 8:30 PM with facial biometric verification [Doc 1]. "
        "Late library passes allow entry up to 9:30 PM with Chief Warden digital approval [Doc 1]."
    )
    return {"final_answer": ans, "citations": ["kit_campus_hostel_and_mess_guide.docx"]}

print("All LangGraph node functions defined.")



In [ ]:
# Construct and Compile the LangGraph StateGraph
builder = StateGraph(StudentAgentState)

# Add Nodes
builder.add_node("router", router_node)
builder.add_node("condonation_calc", condonation_calc_node)
builder.add_node("course_lookup", course_lookup_node)
builder.add_node("policy_rag", policy_rag_node)

# Add Edges
builder.add_edge(START, "router")

def route_decision(state: StudentAgentState) -> str:
    return state["route"]

builder.add_conditional_edges(
    "router",
    route_decision,
    {
        "condonation_calc": "condonation_calc",
        "course_lookup": "course_lookup",
        "policy_rag": "policy_rag"
    }
)

builder.add_edge("condonation_calc", END)
builder.add_edge("course_lookup", END)
builder.add_edge("policy_rag", END)

# Compile the graph
agent = builder.compile()
print("✓ Successfully compiled LangGraph StateGraph!")



In [ ]:
# Run sample test queries through compiled LangGraph Agent
test_queries = [
    "I have 68% attendance due to illness, what can I do?",
    "Who teaches CS401 and when is the exam?",
    "What is the hostel gate closing time?"
]

for q in test_queries:
    print("=" * 60)
    print(f"[STUDENT QUERY]: {q}")
    result = agent.invoke({"query": q})
    print(f"[DETECTED ROUTE]: {result['route']}")
    print(f"[AGENT RESPONSE]:\n{result['final_answer']}")
    print()

